**Imports and setup** for drawing training ROIs, including the interactive hsiViewer.

In [ ]:
import numpy as np
import spectral
from hsiViewer import hsi_viewer_ROI as hvr

# --- Load configuration (paths + parameters live in config.yaml) ---
# config.yaml and the paths inside it are relative to the repo root, but this
# notebook lives in notebooks/. Walk up to the repo root and resolve every
# configured path against it, so this works whether Jupyter is launched from
# the repo root or from notebooks/.
import yaml
from pathlib import Path
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
with open(REPO_ROOT / 'config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)
for _section in ('paths',):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)

## Open the Image

**Open the reflectance image** you want to label. Set its path in `config.yaml`.

In [ ]:
# Reflectance image to draw training ROIs on.
# Default (leave BOTH reflectance_image keys blank in config.yaml): notebook 02's
# output for raw_image, i.e. <raw_image>_ref.img / <raw_image>_ref.hdr. Set the
# two keys only to point notebook 03 at a reflectance image that was NOT produced
# from raw_image above (e.g. one supplied separately). If a key is set but
# disagrees with the raw_image + _ref default, warn rather than silently open the
# wrong cube.
import warnings
_default_hdr = CONFIG['paths']['raw_image'] + '_ref.hdr'
_default_img = CONFIG['paths']['raw_image'] + '_ref.img'
reflectance_image_hdr = CONFIG['paths']['reflectance_image_hdr'] or _default_hdr
reflectance_image     = CONFIG['paths']['reflectance_image']     or _default_img
if CONFIG['paths'].get('reflectance_image') and \
        Path(reflectance_image).stem != Path(_default_img).stem:
    warnings.warn(
        'reflectance_image (' + reflectance_image + ') does not match the default '
        + _default_img + ' derived from raw_image. Opening the configured value; '
        "clear the reflectance_image keys in config.yaml to use notebook 02's output.")
assert Path(reflectance_image_hdr).stem == Path(reflectance_image).stem, \
    "reflectance_image and reflectance_image_hdr name different cubes — check config.yaml"
# Read the image
im = spectral.envi.open(reflectance_image_hdr, reflectance_image)
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

**Interactive.** Opens the hsiViewer ROI tool. Draw labeled training ROIs, naming them with the **same convention as the ASD library** (e.g. `Ammo_bre_...`) so the classifier can read their labels. Save them — these `.pkl` files are the input to the `upwins-veg-classifier` training notebook.

In [ ]:
hvr.viewer(im, stretch=[0,100], rotate=True)